# Imports and Configs

In [ ]:
# Install klib and flash libraries
!pip install klib git+https://github.com/flash-lib/flash.git -q

In [ ]:
# Standard Libraries
import toml

# Data Manipulation
import numpy as np
import pandas as pd

# Data Preprocessing
import klib
import flash as fz
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import KNNImputer, SimpleImputer

In [ ]:
# Change directory to current working directory
%cd /content/drive/MyDrive/Projects/loan-sanction-prediction

In [ ]:
# Load the configurations
with open("config/config.toml", "r") as file:
    config_data = toml.load(file)

num_cols, cat_cols = config_data['num']['cols'], config_data['cat']['cols']
num_cols_with_nan, cat_cols_with_nan = config_data['num']['nan'], config_data['cat']['nan']

# Data Cleaning Steps:

#### 1. Handle Missing Values:

- Numerical features: Use KNN Imputer or Iterative Imputer.
- Categorical features: Use classifier models to predict missing values.

#### 2. Adjust Data Types:

- `applicant_income`: `float`
- `loan_amount_term` and `credit_history`: `int`, then `str`
- categorical features: `category`

#### 3. Handle Outliers:

- Use Tree-Based Models. Because, Tree-Based Models are less sensitive to outliers, so handling outliers may not be necessary if we are using these models.

- Apply feature transformations to make the numerical features more normally distributed, which may reduce the impact of outliers.

- Use IQR-Based Capping to cap outliers to a specific range.

In [ ]:
# Load the dataset
df = pd.read_csv('data/interim/cleaned_train_data_v1.csv')

In [ ]:
# Split the dataset into features and target
X = df.drop('loan_status', axis=1)
y = df['loan_status']

## Handle Missing Values

### Numerical

In [ ]:
# 1. Handle missing values in categorical columns
cat_imputer = SimpleImputer(strategy="constant", fill_value="missing")
X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])

In [ ]:
# OneHotEncoder expects datatypes of every value in a column to be the same
X[cat_cols] = X[cat_cols].astype(str)

In [ ]:
# Store encoded feature names before encoding
unique_values_in_cols = {}
for col in cat_cols:
    encoded_columns = []
    for value in X[col].unique():
        encoded_columns.append(f"{col}_{value}")
    unique_values_in_cols[col] = encoded_columns

In [ ]:
# 2. One-Hot Encode categorical features
encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
encoded_X_data = encoder.fit_transform(X[cat_cols])
encoded_X_df = pd.DataFrame(encoded_X_data, columns=encoder.get_feature_names_out())

# Concatenating encoded categorical features with the rest of the X
X = pd.concat([X.drop(columns=cat_cols), encoded_X_df], axis=1)

In [ ]:
# 3. Impute missing values in numerical features using KNN imputer
knn_imputer = KNNImputer(n_neighbors=5)
X = pd.DataFrame(knn_imputer.fit_transform(X), columns=X.columns)

In [ ]:
# 4. Assign the missing value imputed numerical features back to df
df[num_cols_with_nan] = X[num_cols_with_nan]

### Categorical

In [ ]:
def advanced_categorical_imputer(X, y, clf_model):
    X, y = X.copy(), y.copy() # Avoid modifying the original X and y

    y_notna = y.notna() # Create a mask for non-missing values in y

    # Split the data into training (non-missing) and test (missing) data
    X, X_test = X[y_notna], X[~y_notna]
    y_train, y_test = y[y_notna], y[~y_notna]

    # Label encoding the target feature
    le = LabelEncoder()
    y_train = le.fit_transform(y_train)

    clf_model.fit(X, y_train) # Train the model

    y_pred = clf_model.predict(X_test) # Predict on the test data (missing values)

    # Inverse transform the predicted values to original labels
    y_pred_inverse = le.inverse_transform(y_pred)

    y[y_test.index] = y_pred_inverse # Impute the missing target values

    return y, clf_model, le

In [ ]:
clf_models = {}
label_encoders = {}
for col in cat_cols_with_nan:
    df[col], clf_models[col], label_encoders[col] = advanced_categorical_imputer(
        X.drop(columns=unique_values_in_cols[col], errors='ignore'), df[col],
        ExtraTreesClassifier(random_state=42))

In [ ]:
def check_missing_values(df):
    if df.isna().any().any():
        print("There are still missing values in the DataFrame.")
    else:
        print("There are no missing values left in the DataFrame.")

# Test
check_missing_values(df)

## Data type adjustments


In [ ]:
df['applicant_income'] = df['applicant_income'].astype(float)
df['loan_amount_term'] = df['loan_amount_term'].astype(int).astype(object)
df['credit_history'] = df['credit_history'].astype(int).astype(object)

# Test
print(df.dtypes)

# Export

In [ ]:
# Export dataset
fz.export(df, 'data/interim/cleaned_train_data_v2.csv', force_overwrite=True)

# Notes:

## Example 1: Series with Integer Index

```python
m = pd.Series([10, 20, 30], index=[0, 1, 2])

print(m.loc[1])  # Accesses the value with label 1, returns 20
print(m[1])      # Also accesses the value with label 1, returns 20
print(m.iloc[1]) # Accesses the second position (index 1), returns 20
```

## Example 2: Series with Non-integer Index

```python
n = pd.Series([10, 20, 30], index=['a', 'b', 'c'])

print(n.loc['b'])   # Label-based indexing, returns 20
print(n[1])          # Will not work in future for Series with non-integer labels
print(n.iloc[1])     # Position-based indexing, returns 20
```

### Future Warning:
```
  <ipython-input-244-b47552e90ea2>:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated.
  In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior).
  To access a value by position, use `ser.iloc[pos]`
  print(n[1]) # Also accesses the value with label 1, returns 20
```

### Important Notes:
1. `loc[]` is for label-based indexing, and it should be used when the index labels are non-integer (e.g., 'a', 'b', 'c').
2. `iloc[]` is for position-based indexing, and it should be used when referring to the position (e.g., 1st, 2nd, 3rd element).
3. Using `[]` with integers will be deprecated in future versions for Series with non-integer labels. It will raise an error.
4. Always use `.iloc[]` for position-based indexing and `.loc[]` for label-based indexing to future-proof your code.
